# Predicción del cambio de perclorato con Ridge y rezagos químicos

Copia enfocada en ClO4 de cristales sin refinar y refinados. El objetivo es delta(t→t+2 h) = valor(t+2 h) − valor(t), con predictores químicos de t, t−2 h, t−4 h y t−6 h.

La evaluación separa cronológicamente el 80/20 y selecciona alpha mediante GridSearchCV temporal, con imputación y escalado dentro de cada fold. El baseline de delta cero equivale a mantener el nivel actual.

## 1. Configuración

Dependencias: pandas, numpy, scikit-learn y openpyxl. Cambia RUTA_ARCHIVO si el Excel está en otra ubicación.

In [2]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RUTA_EXCEL_ESCRITORIO = Path("C:/Users/SrLeo/Desktop/SQM/Predicción efecto de analitos/tabla_limpia_valores_reparada.xlsx")
RUTA_EXCEL_RELATIVA = Path("tabla_limpia_valores_reparada.xlsx")
RUTA_ARCHIVO = next((p.resolve() for p in (RUTA_EXCEL_RELATIVA, RUTA_EXCEL_ESCRITORIO) if p.exists()), RUTA_EXCEL_ESCRITORIO)
HOJA_DATOS = "Sheet1"
COLUMNA_TIEMPO = "Fecha_hora"
HORIZONTE_HORAS = 2
FRACCION_ENTRENAMIENTO = 0.80
ALPHAS = np.logspace(-2, 5, 25)  # rango ampliado porque la búsqueda anterior llegaba a 1000

if not RUTA_ARCHIVO.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {RUTA_ARCHIVO}")

## 2. Carga y preparación temporal

La detección de nombres tolera tildes y la errata refinal del encabezado ClO4. Solo se leen los datos de Sheet1; las hojas de trazabilidad, incidencias y resumen no se usan como variables.

In [3]:
def normalizar_nombre(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip().lower()

raw = pd.read_excel(RUTA_ARCHIVO, sheet_name=HOJA_DATOS)
if COLUMNA_TIEMPO not in raw.columns:
    raise KeyError(f"No existe {COLUMNA_TIEMPO!r}. Columnas: {list(raw.columns)}")

raw[COLUMNA_TIEMPO] = pd.to_datetime(raw[COLUMNA_TIEMPO], errors="coerce")
raw = raw.dropna(subset=[COLUMNA_TIEMPO]).sort_values(COLUMNA_TIEMPO)
if raw[COLUMNA_TIEMPO].duplicated().any():
    raise ValueError("Hay timestamps duplicados; resuélvelos antes de modelar.")

data = raw.set_index(COLUMNA_TIEMPO).sort_index()
data = data.asfreq("h")  # crea horas faltantes con valores vacíos; no inventa mediciones
numeric = data.select_dtypes(include="number").copy()

columnas_normalizadas = {col: normalizar_nombre(col) for col in numeric.columns}
objetivos_sin_refinar = [c for c, n in columnas_normalizadas.items() if n.startswith("cristal sin refin")]
objetivos_refinado = [c for c, n in columnas_normalizadas.items() if n.startswith("cristal refin")]
if not objetivos_sin_refinar or not objetivos_refinado:
    raise ValueError("No se detectaron ambos grupos de objetivos; revisa encabezados.")

# Esta copia solo ajusta los dos analitos de perclorato.
objetivos_sin_refinar = [c for c in objetivos_sin_refinar if "clo4" in columnas_normalizadas[c]]
objetivos_refinado = [c for c in objetivos_refinado if "clo4" in columnas_normalizadas[c]]
if not objetivos_sin_refinar or not objetivos_refinado:
    raise ValueError("No se encontraron las columnas de perclorato refinado y sin refinar.")

# Predictores químicos de ML Refi, SSK y Purga/Recirculación.
# Se excluyen analitos de cristal para evitar usar los objetivos como predictores.
prefijos_quimicos = ("ml_refi", "ssk", "purga_recirculacion", "purga recirculacion")
features = [
    c for c, n in columnas_normalizadas.items()
    if n.startswith(prefijos_quimicos) and c not in objetivos_sin_refinar + objetivos_refinado
]
if not features:
    raise ValueError("No se encontraron predictores con los prefijos químicos esperados.")

print(f"Periodo: {data.index.min()} a {data.index.max()}")
print(f"Filas horarias: {len(data):,}")
print(f"Predictores químicos ({len(features)}): {features}")
print(f"Objetivos sin refinar ({len(objetivos_sin_refinar)}): {objetivos_sin_refinar}")
print(f"Objetivos refinados ({len(objetivos_refinado)}): {objetivos_refinado}")

# Relleno temporal solo hacia delante y solo hasta 2 horas.
X_base = numeric[features].ffill(limit=2)
LAGS_HORAS = (0, 2, 4, 6)
X_todos = pd.concat(
    [X_base.shift(lag).add_suffix(f"_lag{lag}h") for lag in LAGS_HORAS],
    axis=1
)
features_modelo = X_todos.columns.tolist()
y_nivel = numeric[objetivos_sin_refinar + objetivos_refinado]
delta_futuro = y_nivel.shift(-HORIZONTE_HORAS) - y_nivel
delta_futuro.columns = [f"delta__{c}" for c in delta_futuro.columns]

display(delta_futuro.notna().sum().rename("deltas_disponibles").to_frame())

Periodo: 2025-01-01 01:00:00 a 2025-12-16 04:00:00
Filas horarias: 8,380
Predictores químicos (22): ['ML_REFI_NANO3', 'ML_REFI_SO4', 'ML_REFI_K', 'ML_REFI_CA', 'ML_REFI_MG', 'ML_REFI_KCLO4', 'ML_REFI_CLO4IC', 'SSK NANO3', 'SSK CL', 'SSK SO4', 'SSK K', 'SSK MG', 'SSK CA', 'SSK KCLO4 IC', 'SSKCLO4 IC', 'Purga_Recirculacion NaNO3', 'Purga_Recirculación SO4', 'Purga_Recirculacion K', 'Purga_Recirculación Ca', 'Purga_Recirculación Mg', 'Purga_Recirculación KClO4 IC', 'Purga_Recirculación ClO4 IC']
Objetivos sin refinar (1): ['Cristal sin refinal ClO4']
Objetivos refinados (1): ['Cristal refinado ClO4']


,deltas_disponibles
delta__Cristal sin refinal ClO4,3789
delta__Cristal refinado ClO4,3883


## 3. Entrenamiento y evaluación cronológica

Se ajusta un modelo para el perclorato sin refinar y otro para el refinado. La búsqueda usa MAE. El baseline de delta cero predice que el nivel se mantiene igual durante las dos horas.

In [4]:
resultados = []
predicciones = []
coeficientes = []
modelos = {}

# Un único corte de calendario para todos los objetivos.
# Se purgan del entrenamiento las etiquetas cuyo horizonte cruza el corte.
corte_temporal = data.index[int(len(data) * FRACCION_ENTRENAMIENTO)]
duracion_horizonte = pd.Timedelta(hours=HORIZONTE_HORAS)

def crear_folds_por_fecha(indice, n_splits=5, horizonte=duracion_horizonte):
    """Folds expansivos con bloques de validación de igual duración y purga del horizonte."""
    indice = pd.DatetimeIndex(indice)
    bordes = pd.date_range(
        start=indice.min(),
        end=indice.max() + pd.Timedelta(hours=1),
        periods=n_splits + 2
    )
    folds = []
    for k in range(n_splits):
        inicio_validacion = bordes[k + 1]
        fin_validacion = bordes[k + 2]
        mascara_train = (indice + horizonte) <= inicio_validacion
        mascara_validacion = (indice >= inicio_validacion) & (indice < fin_validacion)
        idx_train = np.flatnonzero(mascara_train)
        idx_validacion = np.flatnonzero(mascara_validacion)
        if len(idx_train) and len(idx_validacion):
            folds.append((idx_train, idx_validacion))
    if len(folds) < 2:
        raise ValueError("No se pudieron formar suficientes folds temporales por fecha.")
    return folds

for grupo, targets_grupo in [("Sin refinar", objetivos_sin_refinar), ("Refinado", objetivos_refinado)]:
    for target in targets_grupo:
        y_completo = delta_futuro[f"delta__{target}"]
        validos = y_completo.notna()
        X_validos = X_todos.loc[validos]
        y_validos = y_completo.loc[validos]

        # Entrenamiento antes del corte; cada etiqueta debe terminar como máximo en el corte.
        mascara_train = (X_validos.index + duracion_horizonte) <= corte_temporal
        mascara_test = X_validos.index >= corte_temporal
        X_train, y_train = X_validos.loc[mascara_train], y_validos.loc[mascara_train]
        X_test, y_test = X_validos.loc[mascara_test], y_validos.loc[mascara_test]

        if len(y_train) < 20 or len(y_test) == 0:
            print(f"Se omite {target}: datos insuficientes antes o después del corte.")
            continue

        cv_folds = crear_folds_por_fecha(X_train.index)
        pipeline = make_pipeline(
            SimpleImputer(strategy="median", add_indicator=False),
            StandardScaler(),
            Ridge()
        )
        busqueda = GridSearchCV(
            estimator=pipeline,
            param_grid={"ridge__alpha": ALPHAS},
            scoring="neg_mean_absolute_error",
            cv=cv_folds,
            refit=True,
            n_jobs=-1
        )
        busqueda.fit(X_train, y_train)
        modelo = busqueda.best_estimator_
        pred = modelo.predict(X_test)

        mae_ridge = mean_absolute_error(y_test, pred)
        mae_baseline = mean_absolute_error(y_test, np.zeros_like(y_test))
        resultados.append({
            "grupo": grupo, "objetivo": target,
            "fecha_inicio_test": X_test.index.min(),
            "fecha_fin_test": X_test.index.max(),
            "n_train": len(X_train), "n_test": len(X_test),
            "alpha_elegido": busqueda.best_params_["ridge__alpha"],
            "MAE_CV": -busqueda.best_score_,
            "MAE_Ridge": mae_ridge,
            "MAE_baseline_delta_0": mae_baseline,
            "mejora_MAE_vs_baseline": mae_baseline - mae_ridge
        })
        modelos[target] = modelo

        predicciones.append(pd.DataFrame({
            "grupo": grupo, "objetivo": target,
            "delta_real": y_test, "delta_predicho": pred,
            "nivel_actual": y_nivel.loc[y_test.index, target],
            "nivel_futuro_real": y_nivel[target].shift(-HORIZONTE_HORAS).loc[y_test.index],
            "nivel_futuro_predicho": y_nivel.loc[y_test.index, target] + pred
        }, index=y_test.index).rename_axis("timestamp_t"))

        nombres_transformados = modelo.named_steps["simpleimputer"].get_feature_names_out(features_modelo)
        coeficientes.append(pd.DataFrame({
            "grupo": grupo, "objetivo": target,
            "predictor_transformado": nombres_transformados,
            "coeficiente_estandarizado": modelo.named_steps["ridge"].coef_
        }))

if not resultados:
    raise RuntimeError("No se ajustaron modelos; revisa la cobertura de los objetivos y el horizonte.")

metricas = pd.DataFrame(resultados).sort_values(["grupo", "objetivo"]).reset_index(drop=True)
predicciones_test = pd.concat(predicciones).sort_index() if predicciones else pd.DataFrame()
coeficientes_modelos = pd.concat(coeficientes, ignore_index=True) if coeficientes else pd.DataFrame()

display(metricas)

,grupo,objetivo,fecha_inicio_test,fecha_fin_test,n_train,n_test,alpha_elegido,MAE_CV,MAE_Ridge,MAE_baseline_delta_0,mejora_MAE_vs_baseline
0,Refinado,Cristal refinado ClO4,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,6.267031,6.121824,6.088161,-0.033663
1,Sin refinar,Cristal sin refinal ClO4,2025-10-08 19:00:00,2025-12-15 23:00:00,3050,739,51089.697745,83.532827,112.280420,111.958733,-0.321686


## 4. Revisión de resultados

MAE Ridge menor que el baseline indica mejora en el conjunto de prueba para ese analito. El nivel predicho se reconstruye como valor(t) + delta_predicho.

In [5]:
display(predicciones_test.head(20))

coeficientes_ordenados = coeficientes_modelos.assign(
    magnitud=lambda tabla: tabla["coeficiente_estandarizado"].abs()
).sort_values(["objetivo", "magnitud"], ascending=[True, False])

display(coeficientes_ordenados.drop(columns="magnitud").groupby("objetivo", sort=False).head(310))

,grupo,objetivo,delta_real,delta_predicho,nivel_actual,nivel_futuro_real,nivel_futuro_predicho
timestamp_t,,,,,,,
2025-10-08 19:00:00,Sin refinar,Cristal sin refinal ClO4,-8.0,8.197191,99.0,91.0,107.197191
2025-10-08 21:00:00,Sin refinar,Cristal sin refinal ClO4,3.0,6.311154,91.0,94.0,97.311154
2025-10-08 23:00:00,Refinado,Cristal refinado ClO4,2.0,0.100309,20.0,22.0,20.100309
2025-10-08 23:00:00,Sin refinar,Cristal sin refinal ClO4,29.0,4.889335,94.0,123.0,98.889335
2025-10-09 01:00:00,Sin refinar,Cristal sin refinal ClO4,59.0,4.188851,123.0,182.0,127.188851
2025-10-09 01:00:00,Refinado,Cristal refinado ClO4,-2.0,0.073101,22.0,20.0,22.073101
2025-10-09 03:00:00,Refinado,Cristal refinado ClO4,0.0,0.047815,20.0,20.0,20.047815
2025-10-09 03:00:00,Sin refinar,Cristal sin refinal ClO4,23.0,4.825579,182.0,205.0,186.825579
2025-10-09 05:00:00,Refinado,Cristal refinado ClO4,0.0,0.043504,20.0,20.0,20.043504


,grupo,objetivo,predictor_transformado,coeficiente_estandarizado
116,Refinado,Cristal refinado ClO4,ML_REFI_CLO4IC_lag2h,-0.036806
115,Refinado,Cristal refinado ClO4,ML_REFI_KCLO4_lag2h,-0.036372
93,Refinado,Cristal refinado ClO4,ML_REFI_KCLO4_lag0h,-0.031119
94,Refinado,Cristal refinado ClO4,ML_REFI_CLO4IC_lag0h,-0.030972
105,Refinado,Cristal refinado ClO4,Purga_Recirculacion K_lag0h,-0.028546
...,...,...,...,...
55,Sin refinar,Cristal sin refinal ClO4,SSK MG_lag4h,0.014355
58,Sin refinar,Cristal sin refinal ClO4,SSKCLO4 IC_lag4h,-0.007275
56,Sin refinar,Cristal sin refinal ClO4,SSK CA_lag4h,0.005933
57,Sin refinar,Cristal sin refinal ClO4,SSK KCLO4 IC_lag4h,-0.005611


## 5. Guardar resultados (opcional)

Guarda métricas, predicciones de prueba y coeficientes en una subcarpeta junto al Excel.

In [6]:
CARPETA_SALIDA = RUTA_ARCHIVO.parent / "resultados_ridgecv_delta"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
metricas.to_csv(CARPETA_SALIDA / "metricas_ridgecv.csv", index=False, encoding="utf-8-sig")
predicciones_test.to_csv(CARPETA_SALIDA / "predicciones_test_delta.csv", encoding="utf-8-sig")
coeficientes_modelos.to_csv(CARPETA_SALIDA / "coeficientes_estandarizados.csv", index=False, encoding="utf-8-sig")
print(f"Archivos guardados en: {CARPETA_SALIDA}")

Archivos guardados en: /mnt/c/Users/SrLeo/Desktop/SQM/Predicción efecto de analitos/resultados_ridgecv_delta
